# Geometric EEG SSL — Experiment Notebook (MacBook / MPS)

**Purpose:** Run the full eval suite (E1, E2 a/b/c, E5, E7) locally on Apple Silicon (MPS),
without consuming Colab GPU quota.

**Prereqs:**
- Project deps already installed in the active Python env (`mne`, `moabb`,
  `scikit-learn`, `pyyaml`, `scipy`, `torch` with MPS).
- Pretrain checkpoints either already in `runs/pretrain/{variant}_full/`
  or available in the local zips `runs-20260524T025444Z-3-00{1,2}.zip` at
  the repo root. Cell 3 extracts them if missing.
- MNE data cached under `~/mne_data` (the MNE default).

**Note:** MPS is ~5–10× slower than a Colab T4 for these probes but is
free. Each E-script runs the probe head only, not pretraining.

## 1. Paths and environment

In [3]:
import os, sys, pathlib

REPO_DIR = str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
assert (pathlib.Path(REPO_DIR) / 'src').is_dir(), f'REPO_DIR looks wrong: {REPO_DIR}'
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')

RUNS_DIR = f'{REPO_DIR}/runs/pretrain'
EVAL_LOG_DIR = f'{REPO_DIR}/runs/eval/macbook'
os.makedirs(EVAL_LOG_DIR, exist_ok=True)

# Optional preprocessing cache (set if you have one; otherwise eval recomputes).
CACHE_ROOT = os.environ.get('EEG_CACHE_DIR', f'{REPO_DIR}/cache')
if os.path.isdir(CACHE_ROOT):
    os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
    print(f'Preprocessing cache → {CACHE_ROOT}')
else:
    print(f'No cache at {CACHE_ROOT} — eval will recompute preprocessing.')

# MNE_DATA defaults to ~/mne_data on macOS; don't override unless needed.
print(f'REPO_DIR       = {REPO_DIR}')
print(f'RUNS_DIR       = {RUNS_DIR}')
print(f'EVAL_LOG_DIR   = {EVAL_LOG_DIR}')
print(f"MNE_DATA       = {os.environ.get('MNE_DATA', '(default ~/mne_data)')}")

No cache at /Users/zhou/Desktop/geometric-eeg-ssl/cache — eval will recompute preprocessing.
REPO_DIR       = /Users/zhou/Desktop/geometric-eeg-ssl
RUNS_DIR       = /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain
EVAL_LOG_DIR   = /Users/zhou/Desktop/geometric-eeg-ssl/runs/eval/macbook
MNE_DATA       = (default ~/mne_data)


## 2. Device check (MPS)

In [1]:
import torch

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'
print(f'torch {torch.__version__} | device = {DEVICE}')

torch 2.8.0 | device = mps


## 3. Extract checkpoints from local zips (if missing)

The two Colab zips contain `runs/{variant}_full/epoch_*.pt`. Locally the
eval scripts look under `runs/pretrain/{variant}_full/`, so we extract
and remap the prefix.

In [4]:
import zipfile, glob, os

VARIANTS = ['g1_full', 'g2_full', 'g3_full', 'codex_full', 'chind_full']
ZIPS = [
    f'{REPO_DIR}/runs-20260524T025444Z-3-001.zip',
    f'{REPO_DIR}/runs-20260524T025444Z-3-002.zip',
]

def has_ckpts(variant):
    return bool(glob.glob(f'{RUNS_DIR}/{variant}/epoch_*.pt'))

missing = [v for v in VARIANTS if not has_ckpts(v)]
if not missing:
    print('All variants already have checkpoints under runs/pretrain/.')
else:
    print(f'Missing checkpoints for: {missing}. Extracting from zips...')
    for zp in ZIPS:
        if not os.path.exists(zp):
            print(f'  skip (not found): {zp}')
            continue
        with zipfile.ZipFile(zp) as z:
            for name in z.namelist():
                # zip layout: runs/{variant}_full/epoch_*.pt → runs/pretrain/{variant}_full/epoch_*.pt
                if not name.startswith('runs/') or not name.endswith('.pt'):
                    continue
                rel = name[len('runs/'):]
                variant = rel.split('/', 1)[0]
                if variant not in missing:
                    continue
                dest_dir = f'{RUNS_DIR}/{variant}'
                os.makedirs(dest_dir, exist_ok=True)
                dest = f'{dest_dir}/{os.path.basename(rel)}'
                if os.path.exists(dest):
                    continue
                with z.open(name) as src, open(dest, 'wb') as out:
                    out.write(src.read())
                print(f'  + {dest}')
    print('Done.')

Missing checkpoints for: ['codex_full', 'chind_full']. Extracting from zips...
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/codex_full/epoch_0079.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/codex_full/epoch_0029.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/chind_full/epoch_0019.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/codex_full/epoch_0049.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/codex_full/epoch_0089.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/chind_full/epoch_0079.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/codex_full/epoch_0059.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/chind_full/epoch_0039.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/codex_full/epoch_0069.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/codex_full/epoch_0009.pt
  + /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/codex_full/epoch_0099.pt
  + /Users/zhou/Desktop/geomet

## 4. Checkpoint inventory (sanity check)

In [5]:
import glob, os

LABELS = [
    ('G1',                 'g1_full'),
    ('G2',                 'g2_full'),
    ('G3',                 'g3_full'),
    ('Transductive Codex', 'codex_full'),
    ('Channel-Indep',      'chind_full'),
]

print(f"{'Variant':<25} {'Latest checkpoint'}")
print('-' * 60)
for label, sub in LABELS:
    ckpts = sorted(glob.glob(f'{RUNS_DIR}/{sub}/epoch_*.pt'))
    if ckpts:
        print(f'  {label:<23} {os.path.basename(ckpts[-1])}  ({len(ckpts)} saved)')
    else:
        print(f'  {label:<23} (no checkpoints)')

Variant                   Latest checkpoint
------------------------------------------------------------
  G1                      epoch_0099.pt  (10 saved)
  G2                      epoch_0099.pt  (10 saved)
  G3                      epoch_0099.pt  (10 saved)
  Transductive Codex      epoch_0099.pt  (10 saved)
  Channel-Indep           epoch_0099.pt  (10 saved)


## 5. E1 — In-Distribution Linear Probe

PhysioNet MI LOSO: G1 vs. Transductive Codex vs. Channel-Independent.
Fairness anchor for all robustness claims.

In [6]:
!cd {REPO_DIR} && python scripts/run_e1.py --device {DEVICE} --out-tag full 2>&1 | tee {EVAL_LOG_DIR}/e1_log.txt


=== G1 (geometric, score-bias)  (full-scale, epoch_0099.pt) ===

>>> /Users/zhou/Desktop/geometric-eeg-ssl/.venv/bin/python /Users/zhou/Desktop/geometric-eeg-ssl/scripts/probe.py --checkpoint /Users/zhou/Desktop/geometric-eeg-ssl/runs/pretrain/g1_full/epoch_0099.pt --dataset physionet_mi --eval-protocol loso --output /Users/zhou/Desktop/geometric-eeg-ssl/runs/eval/full/e1/g1_full_physionet_mi_loso.json --device mps
device: mps
Discovering n_electrodes from physionet_mi subject 1 ...
  n_electrodes = 64
backbone parameters: 4,770,128  (frozen)
LOSO over 105 subjects on physionet_mi ...
  subject 1: 90 epochs
  subject 2: 90 epochs
  subject 3: 90 epochs
  subject 4: 90 epochs
  subject 5: 90 epochs
  subject 6: 90 epochs
  subject 7: 90 epochs
  subject 8: 90 epochs
  subject 9: 90 epochs
  subject 10: 90 epochs
  subject 11: 90 epochs
  subject 12: 90 epochs
  subject 13: 90 epochs
  subject 14: 90 epochs
  subject 15: 90 epochs
  subject 16: 90 epochs
  subject 17: 90 epochs
  subjec

: 

## 6. E2(a) — Sleep-EDFx cross-session (night 1 → night 2)

In [ ]:
!cd {REPO_DIR} && python scripts/run_e2.py a --device {DEVICE} --out-tag full 2>&1 | tee {EVAL_LOG_DIR}/e2a_log.txt

## 6b. E2(b) — PhysioNet MI LOSO + Wilcoxon

In [ ]:
!cd {REPO_DIR} && python scripts/run_e2.py b --device {DEVICE} --out-tag full 2>&1 | tee {EVAL_LOG_DIR}/e2b_log.txt

## 6c. E2(c) — Cross-montage zero-shot to BCIC-2B

In [ ]:
!cd {REPO_DIR} && python scripts/run_e2.py c --device {DEVICE} --out-tag full 2>&1 | tee {EVAL_LOG_DIR}/e2c_log.txt

## 7. E5 — G1 / G2 / G3 Ablation

In [ ]:
!cd {REPO_DIR} && python scripts/run_e5.py --device {DEVICE} --out-tag full 2>&1 | tee {EVAL_LOG_DIR}/e5_log.txt

## 8. E7 — Channel-Independent Sanity Check

In [ ]:
!cd {REPO_DIR} && python scripts/run_e7.py --device {DEVICE} --out-tag full 2>&1 | tee {EVAL_LOG_DIR}/e7_log.txt

## 9. Full results summary

In [ ]:
import glob, os

print('=== Checkpoint inventory ===')
for label, sub in LABELS:
    ckpts = sorted(glob.glob(f'{RUNS_DIR}/{sub}/epoch_*.pt'))
    status = os.path.basename(ckpts[-1]) if ckpts else '(missing)'
    print(f'  {label:<25} {status}')

print('\n=== Eval logs (last 5 lines each) ===')
for tag in ['e1', 'e2a', 'e2b', 'e2c', 'e5', 'e7']:
    logfile = f'{EVAL_LOG_DIR}/{tag}_log.txt'
    if os.path.exists(logfile):
        lines = open(logfile).readlines()
        print(f'\n--- {tag.upper()} ---')
        print(''.join(lines[-5:]).rstrip())
    else:
        print(f'\n--- {tag.upper()} --- (not run yet)')